In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv("dataset_100k.csv")
df

,themes,solution,epd,rating,rating_dev,to_move,cp_eval,move1cp,nodes,move1multiPV,...,triangleMate,balestraMate,collinearMove,killBoxMate,anastasiaMate,blindSwineMate,castling,mateIn5,underPromotion,collinear
0,crushing hangingPiece long middlegame,f2g3 e6e7 b2b1 b3c1 b1c1 h6c1,r6k/pp2r2p/4Rp1Q/3p4/8/1N1P2b1/PqP3PP/7K w - -,1935,76,1,742,785,2151415,1,...,0,0,0,0,0,0,0,0,0,0
1,advantage endgame short,d3d6 f8d8 d6d8 f6d8,5rk1/1p3ppp/pq1Q1b2/8/8/1P3N2/P4PPP/3R2K1 b - -,1414,75,-1,468,470,338132,1,...,0,0,0,0,0,0,0,0,0,0
2,advantage endgame rookEndgame short,e7f7 f5e5 e2f1 e5e6,8/5R2/1p2P3/p4r2/P6p/1P3Pk1/4K3/8 b - -,1385,80,-1,243,290,265488,1,...,0,0,0,0,0,0,0,0,0,0
3,advantage middlegame short,b6c5 e2g4 h3g4 d1g4,r2qr1k1/b1p2ppp/p5n1/P1p1p3/4P1n1/B2P2Pb/3NBP1...,1084,74,1,281,277,94038,1,...,0,0,0,0,0,0,0,0,0,0
4,crushing endgame fork short,e4d2 d4e2 g1f1 e2c3,6k1/5p1p/4p3/4q3/3n4/2Q3P1/PP1N1P1P/6K1 b - -,1550,75,-1,549,550,546839,1,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
99046,advantage discoveredAttack middlegame pin very...,e4f5 e5f5 g4g3 f5g5 f3g4 f8f4 g4g5 h6g5,2r2r1k/p5p1/1p5p/2pPqP1P/5NK1/5Q2/P6R/6R1 b - -,1748,77,-1,234,291,185873,1,...,0,0,0,0,0,0,0,0,0,0
99047,crushing endgame short,f4g4 g5e3 g4f4 f8f4,1k3r2/1pb3p1/p3p2p/4P1q1/1PQP2R1/P6P/4N1KP/8 b...,1420,181,-1,423,418,352204,1,...,0,0,0,0,0,0,0,0,0,0
99048,endgame mate mateIn1 oneMove pillsburysMate,e4f3 h7h8,5k2/p6R/4PBp1/5p2/1b1K1P2/5bP1/1P5P/4r3 w - -,1435,76,1,1500,1500,63535,1,...,0,0,0,0,0,0,0,0,0,0
99049,advantage attraction defensiveMove fork middle...,e4e5 h5h2 h1h2 f6g4 h2h1 g4e3 e5d6 e7f8,6r1/pp2kb1p/2pp1n2/4P2r/3P4/2N1Q3/PPP3PP/5R1K ...,2344,97,-1,266,301,285726,1,...,0,0,0,0,0,0,0,0,0,0


In [3]:
remove_attributes = []
for c in df["themes"]:
    remove_attributes.extend(c.split(" "))

In [4]:
y = df["rating"]
X = df.drop(columns=["themes", "solution", "epd", "rating_dev", "rating"] + remove_attributes)
X

,to_move,cp_eval,move1cp,nodes,move1multiPV,move1sel_depth,move1w,move1d,move1l,move2cp,...,white_B,white_R,white_Q,white_K,black_p,black_n,black_b,black_r,black_q,black_k
0,1,742,785,2151415,1,62,1000,0,0,-559,...,0,1,1,1,5,0,1,2,1,1
1,-1,468,470,338132,1,32,1000,0,0,31,...,0,1,1,1,5,0,1,1,1,1
2,-1,243,290,265488,1,31,1000,0,0,-314,...,0,1,0,1,3,0,0,1,0,1
3,1,281,277,94038,1,25,999,1,0,-157,...,2,2,1,1,7,2,2,2,1,1
4,-1,549,550,546839,1,38,1000,0,0,-211,...,0,0,1,1,3,1,0,0,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
99046,-1,234,291,185873,1,43,1000,0,0,-173,...,0,2,1,1,5,0,0,2,1,1
99047,-1,423,418,352204,1,45,1000,0,0,77,...,0,1,1,1,5,0,1,1,1,1
99048,1,1500,1500,63535,1,2,1000,0,0,-225,...,1,1,0,1,3,0,2,1,0,1
99049,-1,266,301,285726,1,32,1000,0,0,-458,...,0,1,1,1,5,1,1,2,0,1


In [5]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=67)
print(X_train.shape)
print(y_train.shape)
print(X_test.shape)
print(y_test.shape)
y_train = (y_train // 100).astype(int)
y_test = (y_test // 100).astype(int)

(69335, 40)
(69335,)
(29716, 40)
(29716,)


In [6]:
group_size = 10000
num_rows = len(X_train)

In [7]:
qid_train = np.repeat(np.arange(np.ceil(num_rows / group_size)), group_size)[:num_rows]

In [8]:
from xgboost import XGBRanker
model = XGBRanker(
    objective='rank:pairwise',
    n_estimators=100,
    learning_rate=0.1
)

In [9]:
model.fit(X_train, y_train, qid=qid_train)

,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'rank:pairwise'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,None
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes fr

In [10]:
y_pred = model.predict(X_test)

In [11]:
y_pred

array([-1.1888739, -3.4049418, -1.5385371, ..., -2.0103855, -3.3798478,
       -2.6878433], shape=(29716,), dtype=float32)

In [12]:
from sklearn.metrics import ndcg_score as ndcg

In [13]:
print(f"NDCG: {ndcg([y_test], [y_pred], k=100)}")

NDCG: 0.7612465836862953


In [14]:
max_bin = y_test.max()
y_test_inverted = max_bin - y_test
y_pred_inverted = -y_pred

In [15]:
print(f"NDCG: {ndcg([y_test_inverted], [y_pred_inverted], k=100)}")

NDCG: 0.8473017350956641


In [16]:
from scipy.stats import kendalltau

In [17]:
corr, _ = kendalltau(y_test[:100], y_pred[:100])
print(f"Kendall’s Tau: {corr:.4f}")

Kendall’s Tau: 0.4354


In [18]:
sorted_indices = np.argsort(y_pred)[::-1]

In [19]:
y_test_top = y_test.iloc[sorted_indices] if hasattr(y_test, 'iloc') else y_test[sorted_indices]
y_pred_top = y_pred[sorted_indices]

# 4. Calculate Tau
corr, _ = kendalltau(y_test_top[:100], y_pred_top[:100])

print(f"Kendall’s Tau (Top 100 Hardest): {corr:.4f}")

Kendall’s Tau (Top 100 Hardest): -0.0025


In [20]:
import numpy as np
from sklearn.base import BaseEstimator, ClassifierMixin
from xgboost import XGBRanker

class RankerToClassifier(BaseEstimator, ClassifierMixin):
    # 1. FORCE scikit-learn to recognize this as a classifier
    _estimator_type = "classifier" 

    def __init__(self, ranker_model, group_size=10000):
        self.ranker_model = ranker_model
        self.group_size = group_size
        self.thresholds_ = []
        self.classes_ = []

    def fit(self, X, y):
        y = np.array(y)
        self.classes_ = np.sort(np.unique(y))
        
        # 2. Safely create integer QIDs
        num_rows = len(X)
        num_groups = int(np.ceil(num_rows / self.group_size)) # Explicit int cast
        qid = np.repeat(np.arange(num_groups), self.group_size)[:num_rows]
        
        # Fit the underlying ranker
        self.ranker_model.fit(X, y, qid=qid)
        
        # Predict on training data to establish distribution thresholds
        train_preds = self.ranker_model.predict(X)
        
        self.thresholds_ = []
        cumulative_pct = 0.0
        
        for cls in self.classes_[:-1]:
            class_prop = np.mean(y == cls)
            cumulative_pct += class_prop
            thresh = np.percentile(train_preds, cumulative_pct * 100)
            self.thresholds_.append(thresh)
            
        return self

    def predict(self, X):
        raw_preds = self.ranker_model.predict(X)
        class_indices = np.searchsorted(self.thresholds_, raw_preds)
        return self.classes_[class_indices]

In [21]:
base_ranker = XGBRanker(
    objective='rank:pairwise',
    n_estimators=100,
    learning_rate=0.1
)

# Wrap it into a classifier
main_model = RankerToClassifier(ranker_model=base_ranker, group_size=10000)

In [22]:
# https://scikit-learn.org/stable/developers/develop.html
from sklearn.base import BaseEstimator, RegressorMixin
from sklearn.tree import DecisionTreeRegressor
from sklearn.base import is_classifier, is_regressor

class HierarchicalEnsembleRegressor(RegressorMixin, BaseEstimator):
    def __init__(self, main_model, submodels, bin_overlap=0.0):
        self.main_model = main_model
        self.submodels = submodels
        self.bin_overlap = bin_overlap
        
    def fit(self, X, y):
        X = X.values if hasattr(X, "values") else np.array(X)
        y = y.values if hasattr(y, "values") else np.array(y)

        self.y_min = np.min(y)
        self.y_max = np.max(y)
        X_bins, y_bins = self.get_bins(X, y)

        for i, c in enumerate(zip(X_bins, y_bins)):
            X_bin, y_bin = c
            self.submodels[i].fit(X_bin, y_bin)

        if is_classifier(self.main_model):
            y_bin = self.target_to_bin(y)
            self.main_model.fit(X, y_bin)
        elif is_regressor(self.main_model):
            self.main_model.fit(X, y)
        else:
            #raise ValueError("main_model is neither a recognized classifier nor a regressor!")
            y_bin = self.target_to_bin(y)
            self.main_model.fit(X, y_bin)
    
    def predict(self, X):
        X = X.values if hasattr(X, "values") else np.array(X)

        y_bin_pred = self.predict_bin(X)
        final_pred = np.zeros(len(X))
    
        for i in range(len(self.submodels)):
            mask = (y_bin_pred == i)
            if np.any(mask):
                final_pred[mask] = self.submodels[i].predict(X[mask])
            
        return final_pred

    def predict_bin(self, X):
        if is_classifier(self.main_model):
            return self.main_model.predict(X)
        elif is_regressor(self.main_model):
            y_pred = self.main_model.predict(X)
            return self.target_to_bin(y_pred)
        else:
            #raise ValueError("main_model is neither a recognized classifier nor a regressor!")
            return self.main_model.predict(X)
    
    def get_bins(self, X, y):
        X = X.values if hasattr(X, "values") else np.array(X)
        y = y.values if hasattr(y, "values") else np.array(y)
        N = len(self.submodels)
        width = (self.y_max - self.y_min) / N
    
        i_max = np.floor(((y - self.y_min) + self.bin_overlap) / width).astype(int)
        i_min = np.ceil(((y - self.y_min) - width - self.bin_overlap) / width).astype(int)
        i_min = np.clip(i_min, 0, N - 1)
        i_max = np.clip(i_max, 0, N - 1)
        
        X_bins = [[] for _ in range(N)]
        y_bins = [[] for _ in range(N)]
        
        for j in range(len(y)):
            for b in range(i_min[j], i_max[j] + 1):
                X_bins[b].append(X[j])
                y_bins[b].append(y[j])
    
        X_bins = [np.array(b) for b in X_bins]
        y_bins = [np.array(b) for b in y_bins]
        
        return X_bins, y_bins

    def target_to_bin(self, y):
        N = len(self.submodels)
        width = (self.y_max - self.y_min) / N
        bins = np.floor((y - self.y_min) / width).astype(int)
        return np.clip(bins, 0, N - 1)
        

In [23]:
from xgboost import XGBRegressor
bins = 4
submodels = [XGBRegressor(n_estimators=100, learning_rate=0.1, eval_metric="mae") for _ in range(bins)]

In [24]:
model = HierarchicalEnsembleRegressor(main_model, submodels, bin_overlap=100)
model.fit(X_train, y_train)

In [25]:
y_true_bins = model.target_to_bin(y_test)
y_pred_bins = model.main_model.predict(X_test)

In [26]:
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
acc = accuracy_score(y_true_bins, y_pred_bins)
print(f"Router Accuracy: {acc:.2%}")

print(classification_report(y_true_bins, y_pred_bins))

Router Accuracy: 49.04%
              precision    recall  f1-score   support

           0       0.51      0.51      0.51      6478
           1       0.53      0.52      0.52     13251
           2       0.46      0.47      0.46      8304
           3       0.31      0.32      0.31      1683

    accuracy                           0.49     29716
   macro avg       0.45      0.45      0.45     29716
weighted avg       0.49      0.49      0.49     29716

